<header style="background-color: rgb(0, 62, 92); color: white; margin-top: 20px; padding:28px; ">
  <img src="../Images/Xlogo.png" alt="logo" width="115" style="float: left;">
  <p style=" text-align: center; font-size: 30px;">   
   <strong> APM_52448_EP - Deep Learning in Finance   </strong></p>
    <p style=" text-align: center; font-size: 30px;"> 
    <strong> Project 1 -  Deep pricing and calibration of the Heston model </strong></p>
  <p style=" text-align: left; font-size: 20px;"> Olivier Féron </p>
</header>


# <font color='red'>PLEASE ENTER YOUR FULL NAMES HERE:</font>

- Romain ETIENNE
- John DOE

<font color='red'>**DEADLINE: March 4, 2026 (5:00 pm)**</font>

<font color='red'>**Please send both pdf and ipynb files with name : Name1_Name2_Project1**</font>


# 0. Setup

In [1]:
# --- Imports & global settings ---
import os, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import norm
from scipy.optimize import minimize
from time import perf_counter
import datetime
import joblib
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
# Training with early stopping on validation RMSE
os.makedirs("Data", exist_ok=True)
file_save_best_model = "Data/best_mlp.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED=0
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1 The Heston stochastic volatility model


The Heston Model (Heston, 1993) under the risk-neutral measure $\mathbb{Q}$:

\begin{equation}
  \begin{aligned}
    \frac{dS_t}{S_t} &= r\, dt + \sqrt{v_t}\, dW_t,\\
    dv_t &= \alpha(\beta - v_t)\, dt + \delta \sqrt{v_t}\, dB_t,\\
    d\langle W,B\rangle_t &= \rho\, dt.
  \end{aligned}
\end{equation}

We calibrate the parameter vector $\Theta=(v_0,\alpha,\beta,\delta,\rho)$.

The Heston model is well adapted to fit the volatiliy smile: Heston model's parameters can adapt lots of smile shapes.






We consider **European call options** with payoff $(S_T-K)^+$.  
In this project, calibration is performed **on call prices** (as in your Chapter 3 tutorials), not on implied volatilities.

You will use a neural network surrogate fast pricer:
$$
(\Theta, K) \mapsto C_{\text{Heston}}(K,T)
$$
for a **single maturity** $T$ (same setting as your market dataset).


# Your task

Inspired by your Chapter 3 notebooks (FastPricing_BS_Surrogate and Calibration_NN):

1) Generate simulations of the Heston model
2) Implement the Heston pricer (via Monte Carlo) to generate synthetic training data
3) Train a deep fast pricer on the synthetic data and check performance
4) Train a deep fast pricer
5) Analyze the results and find out what was wrong in the procedure

# 2 Monte Carlo simulation under the Heston model

In this project, we simulate asset price trajectories under the Heston stochastic volatility model
in order to compute option prices via Monte Carlo.


## 2.2 Time discretization

We discretize time:

$$
t_n = n \Delta t, \quad \Delta t = \frac{T}{N}.
$$

We consider correlated Brownian increments:

$$
\begin{aligned}
\Delta W_n &= \sqrt{\Delta t}\, Z_1^n, \\
\Delta B_n &= \sqrt{\Delta t}\left( \rho Z_1^n + \sqrt{1-\rho^2} Z_2^n \right).
\end{aligned}
$$

with $ Z_1^n, Z_2^n \sim \mathcal{N}(0,1) $


---

## 2.3 Variance discretization

A simple Euler scheme gives:

$$
v_{n+1} = v_n + \alpha(\beta - v_n)\Delta t
          + \delta \sqrt{v_n}\, \Delta B_n.
$$

However, this may produce negative values. A common practical fix is the **Full Truncation Euler scheme**:

$$
v_{n+1} = v_n + \alpha(\beta - v_n^+)\Delta t
          + \delta \sqrt{v_n^+}\, \Delta B_n,
$$

with $v_n^+ = \max(v_n, 0).$


---

## 2.4 Stock price discretization

Using a log-Euler scheme:

$$
S_{n+1} =
S_n \exp\left(
(r - \tfrac{1}{2} v_n^+)\Delta t
+ \sqrt{v_n^+}\, \Delta W_n
\right).
$$


<span style="color: red;">**2.1** Implement a function that generates trajectories from the Heston model.</span>


In [ ]:
def simulate_heston(S0,v0,r,alpha,beta,delta,rho,T,n_steps,n_paths):
    """
    Simulate trajectories under the Heston model:
        dS_t / S_t = r dt + sqrt(v_t) dW_t
        dv_t = alpha (beta - v_t) dt + delta sqrt(v_t) dB_t
        d<W,B>_t = rho dt
    using Full Truncation Euler scheme.

    Outputs
    -------
    S : array (n_paths, n_steps+1)
    v : array (n_paths, n_steps+1)
    """

    dt = T / n_steps

    S_paths = np.zeros((n_paths, n_steps + 1))
    v_paths = np.zeros((n_paths, n_steps + 1))

    S_paths[:, 0] = S0
    v_paths[:, 0] = v0

    for t in range(n_steps):
        # YOUR CODE HERE

    return S_paths, v_paths



# Parameters
S0 = 100
v0 = 0.04
r = 0.02
alpha = 2.0
beta = 0.04
delta = 0.6
rho = -0.7
T = 1.0

n_steps = 250
n_paths = 20

# Simulate
S_paths, v_paths = simulate_heston(
    S0, v0, r,
    alpha, beta, delta, rho,
    T, n_steps, n_paths,
)

# Time grid
t_grid = np.linspace(0, T, n_steps + 1)

# Plot stock paths
plt.figure(figsize=(10,4))
for i in range(n_paths):
    plt.plot(t_grid, S_paths[i], lw=1)
plt.title("Heston model - Stock price trajectories")
plt.xlabel("Time")
plt.ylabel("S_t")
plt.grid(alpha=0.3)
plt.show()

# Plot variance paths
plt.figure(figsize=(10,4))
for i in range(n_paths):
    plt.plot(t_grid, v_paths[i], lw=1)
plt.title("Heston model - Variance trajectories")
plt.xlabel("Time")
plt.ylabel("v_t")
plt.grid(alpha=0.3)
plt.show()

# 3. Call option price and dataset generation

Now it is possible to approximate call option prices by Monte Carlo

<span style="color: red;">**3.1** Implement a function that generates trajectories from the Heston model.</span>

In [ ]:
def heston_call_price_mc(S0, v0, r, alpha, beta, delta, rho,T, K,n_steps,n_paths):
    # YOUR CODE HERE


## 3.1 Dataset generation

The objective is to randomly sample Heston parameters in the training domain 
$\mathcal{D}_{\text{domain}}$:

- $v_0 \in [0.01, 0.12]$
- $\alpha \in [0.3, 4.0]$
- $\beta \in [0.01, 0.12]$
- $\delta \in [0.1, 1.2]$
- $\rho \in [-0.95, -0.05]$
- $S_0 = 274.73$
- $K \in [60, 140]$
- $T  = 0.2$


For each $x \in \mathcal{D}_{\text{domain}}$, we then compute Monte Carlo prices $y$ 
using the Heston simulation scheme implemented previously.

The surrogate model will therefore learn the mapping:

$$
(v_0, \alpha, \beta, \delta, \rho, S_0, K, T)
\;\longmapsto\;
C_{\text{Heston}}(S_0,K,T).
$$

<span style="color: red;"> **3.1** Implement a function `build_dataset(n_samples, n_simulations)` that builds a dataset $(x_i, y_i) \in \mathcal{D}_{\text{domain}}$, for $i = 1, \dots, n_{\text{samples}}$</span>

<span style="color: red;">**3.2** Use this function to generate a dataset of 10,000 samples, and split the dataset into $X_{\text{train}}, y_{\text{train}}, X_{\text{val}}, y_{\text{val}}, X_{\text{test}}, y_{\text{test}}$  with a respective repartition (50%, 10%, 40%).</span>

In [ ]:
# ------------------------------------------
# Dataset builder
# ------------------------------------------

def build_dataset(n_samples, n_paths, n_steps=100):

    X = np.zeros((n_samples, 8))
    y = np.zeros((n_samples, 1))

    for i in range(n_samples):

        # YOUR CODE HERE

        

    return X, y

# Generate dataset
X, y = build_dataset(n_samples=10000,n_paths=5000,n_steps=50)

# ------------------------------------------
# Train / Val / Test split
# ------------------------------------------

def random_split(X, y, train_ratio=0.5, val_ratio=0.1):

    # YOUR CODE HERE

    

(X_train, y_train), (X_val, y_val), (X_test, y_test) = random_split(X, y)

print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

## 3.2 Data normalization

<span style="color: red;"> **3.3** Standardize all the data in an appropriate way</span>


In [ ]:
# YOUR CODE HERE



## 4. Surrogate model: a simple MLP

Now that all is ready to train a Neural Network, we consider a simple MLP surrogate model.

We propose a fully-connected network:
- **input dim:** 5 (features $(v_0, \alpha, \beta, \delta, \rho, S_0, K, T)$ standardized)
- **Hidden layers:** 3 layers with sizes 128 --> 128 --> 64, all with ReLU
- **Output dim:** 1 (scaled (standardized) price)
- **Loss function:** MSE (nn.MSELoss)
- **Optimizer:** Adam (Learning rate 1e$^{-3}$

<span style="color: red;">**4.1** Build the class MLP with the above characteristics  </span>

<span style="color: red;">**4.2** Implement the function `run_epoch(loader, train)` that runs one epoch</span>
- <span style="color: red;">sets model.train() if train=True otherwise model.eval()</span>
- <span style="color: red;">iterates over the loader, sends</span>
- <span style="color: red;">returns the losses </span>

<span style="color: red;">**4.3** Implement the training procedure on 250 Epochs, with an overfitting control  </span>
- <span style="color: red;">with a patience counter of 20 (if no loss decrease during 20 Epochs, then stop</span>
- <span style="color: red;">save the best model in `Data/best_mlp.pt`</span>

<span style="color: red;">**4.4** Plot the MSE losses (Learning Curves) for the train and val sets  </span>

In [ ]:
# YOUR CODE HERE



## 5. Evaluation (accuracy)

Now the surrogate model is trained, let's evaluate its performance on the test set

<span style="color: red;">**5.1** Load the best model saved in `file_save_best_model` and the scaling parameters with command `joblib.load` and evaluate with `model.eval()`</span>

<span style="color: red;">**5.2** Compute the predicted `y_test_pred` from X_test (test_loader tensor) and unscale to get `y_test_hat`</span>

<span style="color: red;">**5.3** Compute the MAE and RMSE, and give the scatter plot MC price vs. NN price </span>

<span style="color: red;">**5.4** Give the scatter plot BS price vs. NN price </span>

In [ ]:
# YOUR CODE HERE



# 6 Calibration on real data

The objective is to calibrate the Black-Scholes model, i.e. to estimated the implied volatility $\sigma_\text{imp}$.

We propose to estimate $\theta = (v_0, \alpha, \beta, \delta, \rho)$ by minimizing the quadrati error:
\begin{equation}
\sigma_\text{imp} = \arg\min_{\theta} J^{method}(\theta) 
\end{equation}
with 
\begin{equation}
J^{method}(\theta)  = \frac{1}{I}\sum_{i=1}^I \| C^{mkt}_i - C^{NN}_i(\theta) \|^2
\end{equation}
where $C^{NN}_i(\sigma)$ is the price obtained from the NN Surrogate fast pricer. 





### 6.1 First step  : download and prepare the data

In [ ]:
# Load AAPL calls (semicolon-separated)
df = pd.read_csv("AAPL_Call.csv", sep=";", thousands=",")

df.head(), df.columns, df.shape

S0 = 274.73
r = 0.01
# Price = mid (bid/ask) 
C_mkt_series = (df["Bid"] + df["Ask"]) / 2.0
#else:
#C_mkt_series = df["Last Price"]

K_vec     = df["Strike"].to_numpy(dtype=float)
C_mkt_vec = C_mkt_series.to_numpy(dtype=float)


d = datetime.date(2025, 12, 26)
D = datetime.date(2026, 12, 18)
Tmt = (D - d).days / 365.0  # T in year
T_vec = np.full_like(K_vec, Tmt, dtype=float)

print(K_vec, T_vec)

### 6.2 Cost function

<span style="color: red;">**6.1** Build in the code below the function `nn_price_call_heston` that computes the price from the NN Surrogate Fast pricer.</span>

In [ ]:
def build_features_heston(v0, alpha, beta, delta, rho, S0, K, T):
    '''
    Creates the NN input features for the Heston surrogate model.

    All inputs must be broadcastable arrays of same length.
    '''
    v0    = np.asarray(v0, dtype=float)
    alpha = np.asarray(alpha, dtype=float)
    beta  = np.asarray(beta, dtype=float)
    delta = np.asarray(delta, dtype=float)
    rho   = np.asarray(rho, dtype=float)
    S0    = np.asarray(S0, dtype=float)
    K     = np.asarray(K, dtype=float)
    T     = np.asarray(T, dtype=float)
    
    return np.stack([v0, alpha, beta, delta, rho, S0, K, T], axis=1)

def nn_price_call_heston(model, scaler_X, scaler_y, X):
    # YOUR CODE HERE
    

def loss_nn_heston(theta):
    '''
    theta = [v0, alpha, beta, delta, rho]
    '''

    v0, alpha, beta, delta, rho = theta

    X = build_features_heston(
        v0*np.ones_like(K_mkt),
        alpha*np.ones_like(K_mkt),
        beta*np.ones_like(K_mkt),
        delta*np.ones_like(K_mkt),
        rho*np.ones_like(K_mkt),
        S0*np.ones_like(K_mkt),
        K_mkt,
        T_mkt
    )

    C = nn_price_call_heston(model, scaler_X, scaler_y, X).reshape(-1)

    return float(np.mean((C - C_mkt)**2))

### 6.3 Calibration: minimization procedure

<span style="color: red;">**6.2** Implement the minimization procedure to calibrate the Heston model .</span>

<span style="color: red;">**6.3** Compare in the same graph the observed and model prices .</span>




In [ ]:
# ==============================
# Calibration Heston (global)
# ==============================

# Initial guess
theta_0 = np.array([0.04, 1.0, 0.04, 0.5, -0.5])

# Bounds
HES_BOUNDS = [
    (0.001, 0.2),     # v0
    (0.1, 5.0),       # alpha
    (0.001, 0.2),     # beta
    (0.05, 2.0),      # delta
    (-0.999, 0.999)   # rho
]

# Global variables expected by loss_nn_heston
K_mkt = K_vec.copy()
T_mkt = T_vec.copy()
C_mkt = C_mkt_vec.copy()

# Minimisation
#            YOUR CODE HERE



theta_star = res.x
print("Calibrated parameters:")
print("v0    =", theta_star[0])
print("alpha =", theta_star[1])
print("beta  =", theta_star[2])
print("delta =", theta_star[3])
print("rho   =", theta_star[4])
print("Final loss =", res.fun)

C_nn = np.zeros_like(K_vec, dtype=float)

for i, (Ki, Ti) in enumerate(zip(K_vec, T_vec)):

    # YOUR CODE HERE


    

plt.figure(figsize=(8, 4.5))

plt.plot(K_vec, C_nn, 'o-', color='b', lw=1,
         label='NN price (Heston calibrated)')
plt.plot(K_vec, C_mkt_vec, 'x-', color='r', lw=1,
         label='Market price')

plt.xlabel('Strike K')
plt.ylabel('Call price')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

<span style="color: red;">**6.4** Analyze the results. What do yu think the best way to impoved in the code above (numer of simulations, $\mathcal{D}_\text{domain}$; layers in deep NN...) ? </span> You can write your answer in the next markdown box or change the code above to improve the results (or both !)

# References


1) Steven Heston. A Closed-Form Solution for Options with Stochastic Volatility with Applications to Bond and Currency Options. RFS, 1993

2) Course notebooks: FastPricing_BS_Surrogate, Calibration_NN
